In [1]:
import pandas as pd
from kafka import KafkaProducer
from models import ride_serializer, ride_from_row
from time import time

In [2]:
path = "../green_tripdata_2025-10.parquet"
columns = [
    'lpep_pickup_datetime', 
    'lpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'passenger_count',
    'trip_distance',
    'tip_amount',
    'total_amount'
]

df_data = pd.read_parquet(path, columns=columns)


df_data.fillna(0, inplace=True)

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount,total_amount
0,2025-10-01 00:21:47,2025-10-01 00:24:37,247,69,1.0,0.70,1.70,10.00
1,2025-10-01 00:14:03,2025-10-01 00:24:14,66,25,1.0,1.61,2.78,16.68
2,2025-10-01 00:16:44,2025-10-01 00:16:47,244,244,1.0,0.00,2.20,13.20
3,2025-10-01 00:07:36,2025-10-01 00:32:14,95,170,1.0,10.37,11.31,67.85
4,2025-09-30 21:10:29,2025-09-30 21:22:30,82,138,1.0,4.07,6.82,34.12
...,...,...,...,...,...,...,...,...
49411,2025-10-31 23:02:00,2025-11-01 00:28:33,241,61,0.0,20.09,0.00,63.84
49412,2025-10-31 23:51:34,2025-11-01 00:20:58,53,225,0.0,10.11,0.00,34.76
49413,2025-10-31 23:08:00,2025-10-31 23:42:00,7,170,0.0,4.20,10.03,60.17
49414,2025-10-31 23:45:00,2025-11-01 00:08:00,255,25,0.0,4.20,4.86,37.29


In [3]:
df_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 49416 entries, 0 to 49415
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   lpep_pickup_datetime   49416 non-null  datetime64[us]
 1   lpep_dropoff_datetime  49416 non-null  datetime64[us]
 2   PULocationID           49416 non-null  int32         
 3   DOLocationID           49416 non-null  int32         
 4   passenger_count        49416 non-null  float64       
 5   trip_distance          49416 non-null  float64       
 6   tip_amount             49416 non-null  float64       
 7   total_amount           49416 non-null  float64       
dtypes: datetime64[us](2), float64(4), int32(2)
memory usage: 2.6 MB


In [4]:
df_data.head()

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount,total_amount
0,2025-10-01 00:21:47,2025-10-01 00:24:37,247,69,1.0,0.70,1.70,10.00
1,2025-10-01 00:14:03,2025-10-01 00:24:14,66,25,1.0,1.61,2.78,16.68
2,2025-10-01 00:16:44,2025-10-01 00:16:47,244,244,1.0,0.00,2.20,13.20
3,2025-10-01 00:07:36,2025-10-01 00:32:14,95,170,1.0,10.37,11.31,67.85
4,2025-09-30 21:10:29,2025-09-30 21:22:30,82,138,1.0,4.07,6.82,34.12


In [5]:
server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=ride_serializer
)

In [6]:
topic_name = 'green-trips'

t0 = time()

for _, row in df_data.iterrows():
    ride = ride_from_row(row)
    producer.send(topic_name, value=ride)

producer.flush()

t1 = time()

print(f'took {(t1-t0):.2f} seconds')

took 14.17 seconds
